In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

class State(TypedDict):
    text:str


def node_a(state:State):
    print("NODE A")

    return Command(
        goto="node_b",
        update={
            "text":state["text"]+"a"
        }
    )

def node_b(state:State):
    print("NODE B")
# stops the graph here and can be resumed later
    human_response = interrupt("Do you want to go to C or D?(C/D)")

    if human_response=="C":
        return Command(
            goto="node_c",
            update={
                "text":state["text"]+"b1"
            }
        )
    else:
        return Command(
            goto="node_d",
            update={
                "text":state["text"]+"b2"
            }
        )

def node_c(state:State):
    print("NODE C")

    return Command(
        goto="node_d",
        update={
            "text":state["text"]+"c"
        }
    )

def node_d(state:State):
    print("NODE D")

    return Command(
        goto=END,
        update={
            "text":state["text"]+"d"
        }
    )

graph = StateGraph(State)

graph.add_node("node_a",node_a)
graph.add_node("node_b",node_b)
graph.add_node("node_c",node_c)
graph.add_node("node_d",node_d)

graph.set_entry_point("node_a")

app = graph.compile(checkpointer=memory)

config = {
    "configurable":{
        "thread_id":1
    }
}

response = app.invoke({
    "text":""
},config,stream_mode="updates")

response

NODE A
NODE B


[{'node_a': {'text': 'a'}},
 {'__interrupt__': (Interrupt(value='Do you want to go to C or D?(C/D)', id='c1eb54ad9c5016111fea9c22f2198e55'),)}]

In [3]:
print(app.get_state(config).next)

('node_b',)


In [4]:
second_result = app.invoke(Command(resume="D"),config=config,stream_mode="updates")
second_result

NODE B
NODE D


[{'node_b': {'text': 'ab2'}}, {'node_d': {'text': 'ab2d'}}]